# Full-Pol PolSAR Decomposition Tutorial

This notebook demonstrates eleven polarimetric decompositions on **quad-polarization** SAR data
(HH, HV, VH, VV) using GRDL.  Each decomposition provides a physically interpretable
breakdown of the measured scattering.  Spatial averaging is performed once via a selectable
filter (boxcar or Refined Lee) and the resulting $[T_3]$/$[C_3]$ matrices are shared across
all matrix-based algorithms.

**Algorithms demonstrated:**
1. **Pauli decomposition** — coherent, pixel-by-pixel basis transform
2. **FullPolHAalpha** — Cloude-Pottier eigendecomposition (H/A/α)
3. **FreemanDurden3C** — model-based 3-component power decomposition
4. **ModelFree3C (MF3CF)** — model-free 3-component decomposition (Dey et al. 2020)
5. **ModelFree4C (MF4CF)** — model-free 4-component with helix power (Dey et al. 2021)
6. **DegreeOfPolarization** — Barakat scalar polarisation purity (Barakat 1977)
7. **ShannonEntropy** — intensity + polarimetric entropy partition (Morio et al. 2009)
8. **NeumannDecomposition** — de-orientation coherence parameters (Neumann et al. 2010)
9. **PraksParameters** — seven span-normalised target indices (Praks et al. 2009)
10. **TouziDecomposition** — TSVM eigendecomposition (Touzi 2007)
11. **Yamaguchi4C** — 4-component model-based decomposition with helix (Yamaguchi 2005/2011)
12. Vegetation indices (RVI, GRVI, PRVI) — placeholder

**Data:** NISAR L1 RSLC quad-pol chip or synthetic fallback.

> **Dual-pol H/Alpha:** see `dual_pol_decomposition_tutorial.ipynb`.

---

## Theory Overview

### Matrix Filtering

All eigendecomposition and model-based algorithms operate on spatially-averaged polarimetric
matrices.  Two filter modes are available (set `filter_method` in the Configuration cell):

- **`boxcar`** — uniform spatial average over a `window_size × window_size` window.
  Simple and fast; standard in the PolSAR literature.
- **`refined_lee`** — edge-preserving Refined Lee speckle filter applied to the
  single-look coherency matrix $[T_3]$.  Better spatial resolution at feature edges.

$[C_3]$ is derived from filtered $[T_3]$ via the Pauli-to-lexicographic basis change
$[C_3] = D^T [T_3] D$ where:

$$D = \frac{1}{\sqrt{2}}\begin{pmatrix}1 & 0 & 1 \\ 1 & 0 & -1 \\ 0 & \sqrt{2} & 0\end{pmatrix}$$

### Pauli Decomposition

The 2×2 scattering matrix $S$ is expressed in the Pauli basis:

$$S = a P_1 + b P_2 + c P_3$$

$$a = \frac{S_{HH}+S_{VV}}{\sqrt{2}}, \quad b = \frac{S_{HH}-S_{VV}}{\sqrt{2}}, \quad c = \frac{S_{HV}+S_{VH}}{\sqrt{2}}$$

By default this is pixel-by-pixel (single-look); set `pauli_window_size > 1` for optional boxcar channel averaging.
RGB mapping: **R** = $|b|$ (double-bounce), **G** = $|c|$ (volume), **B** = $|a|$ (surface).

### Cloude-Pottier H/A/α (FullPolHAalpha)

Eigendecomposition of $[T_3]$ yields entropy $H$, anisotropy $A$, and mean scattering
angle $\alpha$.  See `dual_pol_decomposition_tutorial.ipynb` for full equations.

### Freeman-Durden 3C

Model-based decomposition of $[C_3]$ into surface ($P_s$), double-bounce ($P_d$),
and volume ($P_v$) scattering contributions:

$$[C_3] = f_s [C_3]^{\text{surface}} + f_d [C_3]^{\text{dbl}} + f_v [C_3]^{\text{vol}}$$

RGB mapping: **R** = $P_d$, **G** = $P_v$, **B** = $P_s$.

**Reference:** Freeman & Durden (1998). IEEE TGRS, 36(3), 963–973.

### Model-Free Decompositions (MF3CF / MF4CF)

Dey et al. derive a rotation angle $\theta_{FP}$ and degree of polarisation $m$ directly
from $[T_3]$ elements, requiring no scene-model assumptions:

$$\theta_{FP} = \frac{1}{2}\arctan\frac{T_{22}+T_{33}-T_{11}}{T_{11}-T_{22}+T_{33}}, \qquad
m = \sqrt{1 - \frac{27 \det[T_3]}{\mathrm{tr}^3[T_3]}}$$

MF4CF adds helix power $P_c$ via ellipticity angle $\tau_{FP}$.

**References:**
- Dey et al. (2020). IEEE GRSL, 17(11).
- Dey et al. (2021). IEEE GRSL.

### Degree of Polarization (Barakat)

Scalar $m \in [0,1]$ measuring polarimetric purity from the eigenvalues of $[T_3]$:

$$m = \sqrt{\max\!\left(1 - \frac{27\det[T_3]}{\operatorname{tr}[T_3]^3},\; 0\right)}$$

$m=0$: fully depolarised; $m=1$: fully polarised (rank-1 $[T_3]$).

**Reference:** Barakat (1977). Optica Acta, 24(5), 537–540.

### Shannon Entropy

Partitions total scene randomness into an intensity term $H_I$ and a polarimetric term $H_P$:

$$H_I = 3\ln\!\left(\frac{e\pi\,\operatorname{tr}[T_3]}{3}\right), \qquad
H_P = \ln\!\left(1 - \frac{27\det[T_3]}{\operatorname{tr}[T_3]^3}\right), \qquad
H = H_I + H_P$$

**Reference:** Morio et al. (2009). IEEE GRSL, 6(4), 728–732.

### Neumann De-orientation Decomposition

Applies a de-orientation rotation $\Phi$ that removes target orientation effects,
then extracts orientation angle $\psi \in [-45°,45°]$, coherence magnitude
$\delta_\text{mod}$, coherence phase $\delta_\text{pha}$, and depolarisation
$\tau \in [0,1]$ from the rotated matrix $[T_3']$.

**Reference:** Neumann et al. (2010). IEEE TGRS, 48(3), 1227–1240.

### Praks Parameters

Seven indices from the span-normalised covariance matrix $M = [C_3]/\operatorname{span}$:
Frobenius norm, scattering predominance, scattering diversity, degree of purity,
depolarisation index, alpha angle, and polarimetric entropy.

**Reference:** Praks et al. (2009). IEEE TGRS, 47(11), 3811–3820.

### Touzi TSVM

Eigendecomposition of $[T_3]$ yielding four physically interpretable parameters per
eigenvector $\vec{k}_i$: scattering type $\alpha_i \in [0°,90°]$, scattering phase
$\phi_i \in (-180°,180°]$, helicity $\tau_i \in (-45°,45°]$, and orientation
$\psi_i \in (-90°,90°]$.  Probability-weighted means summarise the dominant mechanism.

**Reference:** Touzi (2007). IEEE TGRS, 45(9), 2941–2954.

### Yamaguchi 4-Component Decomposition

Extends Freeman-Durden with a helix power component $P_c$ sensitive to urban scene asymmetry:

$$P_\text{span} = P_s + P_d + P_v + P_c$$

Three variants: **y4o** (original 2005), **y4r** (rotation-corrected 2011), **y4s** (extended
volume model).  The per-pixel solver is JIT-compiled with numba when `grdl[polsar]` is installed
(~300–700× faster than an equivalent pure-Python loop).

**References:**
- Yamaguchi et al. (2005). IEEE TGRS, 43(8), 1699–1706.
- Yamaguchi et al. (2011). IEEE TGRS, 49(6), 2251–2258.


In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
import gc

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")


In [ ]:
from pathlib import Path
import gc
import numpy as np
import matplotlib.pyplot as plt

from grdl.IO.sar.nisar import NISARReader
from grdl.image_processing.decomposition import (
    PauliDecomposition, FullPolHAalpha,
    FreemanDurden3C, ModelFree3C, ModelFree4C,
    DegreeOfPolarization, ShannonEntropy,
    NeumannDecomposition, PraksParameters,
    TouziDecomposition, Yamaguchi4C,
    CoherencyMatrix, CovarianceMatrix,
)
from grdl.image_processing.filters import RefinedLeeFilter

try:
    from grdk.viewers.dual_viewer import DualGeoViewer
    HAS_GRDK = True
    try:
        get_ipython().run_line_magic('gui', 'qt6')
    except (NameError, AttributeError):
        pass
except ImportError as exc:
    DualGeoViewer = None
    HAS_GRDK = False
    print(f"Warning: grdk import failed ({exc}). Falling back to matplotlib.")

def show_dual_rgb(left, right, title, left_label='Left', right_label='Right'):
    if HAS_GRDK:
        viewer = DualGeoViewer()
        viewer.set_mode('dual')
        viewer.setWindowTitle(title)
        viewer.set_array(left, pane=0)
        viewer.set_array(right, pane=1)
        viewer.show()
        print(f"GRDK viewer launched: {title}")
        return viewer

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), dpi=100)
    for ax, arr, label in zip(axes, (left, right), (left_label, right_label)):
        img = np.asarray(arr)
        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.moveaxis(img, 0, -1)
        if img.ndim == 3 and img.shape[-1] == 1:
            img = img[..., 0]
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(label)
        ax.axis('off')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    print(f"Matplotlib fallback shown: {title}")
    return fig

print('Imports OK')


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
nisar_file = Path(
    '/data/sar/slc/nisar/l1_rslc/'
    '20260105T055924_20260105T055931/'
    'NISAR_L1_PR_RSLC_009_116_D_054_2005_QPDH_A_20260105T055924_20260105T055931'
    '_X05010_N_P_J_001.h5'
)
frequency     = 'A'
polarizations = 'all'
chip_size     = 1024
chip_center   = [9000, 10000]   # (row, col) tuple; None → auto-center

window_size   = 7     # spatial averaging window for matrix-based decompositions

# --- Matrix filter selection -------------------------------------------
filter_method = 'refined_lee'   # 'boxcar'      → uniform boxcar (CoherencyMatrix / CovarianceMatrix)
                            # 'refined_lee' → edge-preserving Refined Lee filter

# --- Pauli decomposition averaging -------------------------------------
if filter_method == 'refined_lee':
    pauli_window_size = window_size      # Use same size as refined Lee filter for Pauli decomposition averaging
else:
    pauli_window_size = 1      # odd integer >=1; 1 = no averaging, >1 = boxcar pre-average channels

# --- Yamaguchi model (section 10) --------------------------------------
yamaguchi_model = 'y4r'    # 'y4o' original  |  'y4r' rotation-corrected  |  'y4s' extended volume

In [ ]:
# =============================================================================
# Load NISAR quad-pol or generate synthetic fallback
# =============================================================================
def _synthetic_quad_pol(rows=512, cols=512, seed=42):
    """Simple quad-pol SLC with scene regions."""
    rng = np.random.default_rng(seed)
    shh = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    shv = (0.3 * (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols)))).astype(np.complex64)
    svh = shv.copy()
    svv = (rng.standard_normal((rows, cols)) + 1j * rng.standard_normal((rows, cols))).astype(np.complex64)
    # Urban block: enhance double-bounce (HH+VV out-of-phase)
    r0, r1, c0, c1 = rows//8, rows*3//8, cols//8, cols*3//8
    shh[r0:r1, c0:c1] *= 2.0
    svv[r0:r1, c0:c1] *= -2.0
    shv[r0:r1, c0:c1] *= 0.2
    # Vegetation: high cross-pol
    r0, r1, c0, c1 = rows*5//8, rows*7//8, cols*5//8, cols*7//8
    shv[r0:r1, c0:c1] *= 4.0
    svh[r0:r1, c0:c1] *= 4.0
    return shh, shv, svh, svv

if nisar_file.exists():
    reader = NISARReader(nisar_file, frequency=frequency, polarizations=polarizations)
    meta = reader.metadata
    N = chip_size
    if chip_center is not None:
        cr, cc = chip_center
    else:
        cr, cc = meta.rows // 2, meta.cols // 2
    r0 = max(0, cr - N // 2)
    c0 = max(0, cc - N // 2)
    cube = reader.read_chip(r0, r0 + N, c0, c0 + N)
    reader.close()
    pol_index = {cm.polarization: i for i, cm in enumerate(meta.channel_metadata)}
    shh = cube[pol_index['HH']]
    shv = cube[pol_index['HV']]
    svh = cube[pol_index.get('VH', pol_index['HV'])]
    svv = cube[pol_index['VV']]
    print(f'NISAR chip: {shh.shape}')
    del cube
    gc.collect()

else:
    N = chip_size
    shh, shv, svh, svv = _synthetic_quad_pol(rows=N, cols=N)
    print(f'Synthetic quad-pol: {shh.shape}')

In [ ]:
# =============================================================================
# Build filtered [T3] and [C3] matrices
# =============================================================================
# Both matrices are computed once here and reused by all decompositions below.
# filter_method = 'boxcar'      → standard uniform boxcar average
# filter_method = 'refined_lee' → edge-preserving Refined Lee speckle filter
#
# [T3] = Pauli coherency matrix   k = (HH+VV, HH−VV, 2HV)^T / √2
# [C3] = lexicographic covariance k = (HH, √2·HV, VV)^T
# Both share the same span (trace).

channels = np.stack([shh, shv, svh, svv], axis=0)

if filter_method == 'refined_lee':
    rlf = RefinedLeeFilter(kernel_size=window_size)

    # Build single-look [T3] from outer product, then apply Refined Lee filter.
    k0 = (shh + svv) / np.sqrt(2, dtype=np.float32)
    k1 = (shh - svv) / np.sqrt(2, dtype=np.float32)
    k2 = (shv + svh) / np.sqrt(2, dtype=np.float32)
    k = np.stack([k0, k1, k2], axis=0)                    # (3, rows, cols)
    t3_sl = np.einsum('i...,j...->ij...', k, k.conj())    # (3,3,rows,cols) single-look
    t3 = rlf.filter_matrix(t3_sl)
    del t3_sl, k, k0, k1, k2

    # Build the C3 filtered matrix directly (conversion from T3 is not recommended).
    c3 = rlf.filter_channels(shh, shv, svh, svv, matrix_type='C3')

elif filter_method == 'boxcar':
    t3 = CoherencyMatrix(window_size=window_size).compute(channels)
    c3 = CovarianceMatrix(window_size=window_size).compute(channels)

else:
    raise ValueError(f"filter_method must be 'boxcar' or 'refined_lee', got {filter_method!r}")

print(f'filter : {filter_method!r},  window_size={window_size}')
print(f't3: {t3.shape}  dtype={t3.dtype}')
print(f'c3: {c3.shape}  dtype={c3.dtype}')
gc.collect()


---
## 1. Pauli Decomposition — Coherent Scattering Components

Pauli decomposition is a coherent basis transform on the scattering matrix channels.
By default (`pauli_window_size = 1`) it is single-look with no spatial averaging.

Set `pauli_window_size > 1` to apply boxcar pre-averaging to the channels before decomposition.
For non-boxcar filters, pre-filter channels externally before calling `decompose(...)`,
or use `decompose_from_t3(...)` / `decompose_from_c3(...)` when working from matrix products.

In [ ]:
# =============================================================================
# Pauli decomposition
# =============================================================================
pauli = PauliDecomposition(window_size=pauli_window_size)
if filter_method == 'refined_lee':
    pauli_c = pauli.decompose_from_t3(t3)
    # Keys: 'surface', 'double_bounce', 'volume' — complex Pauli coefficients
    print(f"Pauli components: {list(pauli_c.keys())}  (window_size={window_size}), filtered from [T3]")
else:
    pauli_c = pauli.decompose(shh, shv, svh, svv)
    # Keys: 'surface', 'double_bounce', 'volume' — complex Pauli coefficients
    print(f"Pauli components: {list(pauli_c.keys())}  (window_size={pauli_window_size})")

def stretch(x, pct=(2, 98)):
    lo, hi = np.percentile(x, pct)
    return np.clip((x - lo) / max(hi - lo, 1e-8), 0.0, 1.0).astype(np.float32)

# Standard PolSAR Pauli RGB: R=double-bounce, G=volume, B=surface
rgb_pauli = np.stack([
    stretch(np.abs(pauli_c['double_bounce'])),
    stretch(np.abs(pauli_c['volume'])),
    stretch(np.abs(pauli_c['surface'])),
], axis=0)
print(f'Pauli RGB: {rgb_pauli.shape}  (R=double-bounce, G=volume, B=surface)')

In [ ]:
# =============================================================================
# GRDK viewer — Pauli decomposition
# =============================================================================
# Left: raw intensity (R=HH, G=HV, B=VV), Right: Pauli RGB
rgb_raw = np.stack([
    stretch(np.abs(shh)),
    stretch(np.abs(shv)),
    stretch(np.abs(svv)),
], axis=0)

viewer_pauli = show_dual_rgb(
    rgb_raw,
    rgb_pauli,
    f"Pauli (win={pauli_window_size}) — Raw (R=HH,G=HV,B=VV) (L) vs Pauli RGB (R)",
    left_label='Raw intensity RGB',
    right_label='Pauli RGB',
)
print(f'Viewer: Pauli  (window_size={pauli_window_size})')

---
## 2. FullPolHAalpha — Cloude-Pottier Eigendecomposition

Eigendecomposition of the spatially-averaged $[T_3]$ coherency matrix.  The window
size controls the speckle-vs-resolution trade-off.

In [ ]:
# =============================================================================
# FullPolHAalpha
# =============================================================================
fp_ha = FullPolHAalpha(window_size=window_size)
ha_c = fp_ha.decompose_from_t3(t3)
print(f'FullPolHAalpha keys: {list(ha_c.keys())}')
print(f'H   range: [{ha_c["entropy"].min():.3f}, {ha_c["entropy"].max():.3f}]')
print(f'α   range: [{ha_c["alpha"].min():.1f}°, {ha_c["alpha"].max():.1f}°]')
print(f'A   range: [{ha_c["anisotropy"].min():.3f}, {ha_c["anisotropy"].max():.3f}]')

rgb_ha, ha_meta = fp_ha.to_rgb(ha_c)
print(f'RGB: {rgb_ha.shape}  ({" / ".join(cm.name for cm in ha_meta.channel_metadata)})')


In [ ]:
# =============================================================================
# GRDK viewer — H/A/Alpha
# =============================================================================
viewer_ha = show_dual_rgb(
    rgb_pauli,
    rgb_ha,
    f'FullPolHAalpha (win={window_size}) — Pauli (L) vs H/A/α RGB (R)',
    left_label='Pauli RGB',
    right_label='H/A/α RGB',
)
print('Viewer: FullPolHAalpha')

In [ ]:
# =============================================================================
# H-Alpha plane (Cloude-Pottier classification diagram)
# =============================================================================
# Reference: Cloude & Pottier (1997), IEEE TGRS 35(1), Fig. 4
#
# BASIS: [T3] Pauli coherency matrix — k=(HH+VV, HH−VV, 2HV)/√2.
#   alpha = arccos(|e_i[0]|) where e_i[0] is the (HH+VV)/√2 component.
#   alpha≈0°  → surface / odd-bounce (HH+VV dominates)
#   alpha≈45° → volume / random (HV cross-pol dominates)
#   alpha≈90° → double-bounce (HH−VV dominates, dihedral reflector)
#
# For L-band NISAR data over vegetation or urban terrain, alpha=40–85° is
# physically expected: L-band penetrates canopy creating volume scattering
# (alpha≈45–60°) and double-bounce from tree trunks and structures
# (alpha≈60–90°).  This is NOT comparable to the dual-pol alpha, which
# is computed in the [C2] lexicographic (HH, HV) basis.
H_flat     = ha_c['entropy'].ravel()
alpha_flat = ha_c['alpha'].ravel()

mask_ha = np.isfinite(H_flat) & np.isfinite(alpha_flat)
H_valid     = H_flat[mask_ha]
alpha_valid = alpha_flat[mask_ha]

fig, ax = plt.subplots(figsize=(8, 6))

hb = ax.hexbin(H_valid, alpha_valid, gridsize=120, cmap='inferno',
               mincnt=1, extent=[0, 1, 0, 90])
fig.colorbar(hb, ax=ax, label='Pixel count')

# Zone boundaries (Cloude-Pottier 9-zone scheme)
for h_line in [0.5, 0.9]:
    ax.axvline(h_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)
for a_line in [42.5, 47.5]:
    ax.axhline(a_line, color='gray', linewidth=0.8, linestyle='--', alpha=0.7)

zone_labels = {
    (0.25, 21):  'Z9\nSurface',
    (0.25, 45):  'Z8\nDipole',
    (0.25, 69):  'Z7\nDihedral',
    (0.70, 21):  'Z6',
    (0.70, 45):  'Z5\nVegetation',
    (0.70, 69):  'Z4',
    (0.95, 21):  'Z3',
    (0.95, 45):  'Z2',
    (0.95, 69):  'Z1',
}
for (hpos, apos), label in zone_labels.items():
    ax.text(hpos, apos, label, color='cyan', fontsize=7,
            ha='center', va='center', alpha=0.8)

ax.set_xlabel('Entropy (H)')
ax.set_ylabel('Alpha (°)  [T3 Pauli basis]')
ax.set_title('H-α Plane — Cloude-Pottier (full-pol, [T3] Pauli basis)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 90)
plt.tight_layout()
plt.show()

print(f'Valid pixels: {mask_ha.sum():,} / {mask_ha.size:,}')


---
## 3. Freeman-Durden 3C — Model-Based Power Decomposition

Decomposes $[C_3]$ into surface, double-bounce, and volume contributions assuming
specific electromagnetic scattering models for each component.  Works best for
scenes with clear mechanism separation (low or medium rotation).

In [ ]:
# =============================================================================
# FreemanDurden3C
# =============================================================================
fd = FreemanDurden3C(window_size=1)
fd_c = fd.decompose_from_c3(c3)
print(f'FreemanDurden3C keys: {list(fd_c.keys())}')
print(f'Ps range: [{fd_c["surface"].min():.4f}, {fd_c["surface"].max():.4f}]')
print(f'Pd range: [{fd_c["double_bounce"].min():.4f}, {fd_c["double_bounce"].max():.4f}]')
print(f'Pv range: [{fd_c["volume"].min():.4f}, {fd_c["volume"].max():.4f}]')

rgb_fd, fd_meta = fd.to_rgb(fd_c)
print(f'RGB: {rgb_fd.shape}  ({" / ".join(cm.name for cm in fd_meta.channel_metadata)})')


In [ ]:
# =============================================================================
# GRDK viewer — Freeman-Durden
# =============================================================================
viewer_fd = show_dual_rgb(
    rgb_pauli,
    rgb_fd,
    f'FreemanDurden3C (win={window_size}) — Pauli (L) vs FD RGB (R)',
    left_label='Pauli RGB',
    right_label='Freeman-Durden RGB',
)
print('Viewer: FreemanDurden3C')

---
## 4. Model-Free Decompositions — MF3CF and MF4CF

Unlike Freeman-Durden, the model-free decompositions use the degree of polarisation $m$
and the target orientation angle $\theta_{FP}$ computed directly from $[T_3]$, with no
a priori scene model.  MF4CF adds a helix power component $P_c$ via the ellipticity
angle $\tau_{FP}$.

In [ ]:
# =============================================================================
# ModelFree3C (MF3CF)
# =============================================================================
mf3 = ModelFree3C(window_size=window_size)
mf3_c = mf3.decompose_from_t3(t3)
print(f'MF3CF keys: {list(mf3_c.keys())}')

rgb_mf3, mf3_meta = mf3.to_rgb(mf3_c)

# =============================================================================
# ModelFree4C (MF4CF)
# =============================================================================
mf4 = ModelFree4C(window_size=window_size)
mf4_c = mf4.decompose_from_t3(t3)
print(f'MF4CF keys: {list(mf4_c.keys())}')
print(f'Helix Pc range: [{mf4_c["helix"].min():.4f}, {mf4_c["helix"].max():.4f}]')

rgb_mf4, mf4_meta = mf4.to_rgb(mf4_c)
print(f'MF3CF RGB: {rgb_mf3.shape},  MF4CF RGB: {rgb_mf4.shape}')


In [ ]:
# =============================================================================
# GRDK viewer — MF3CF vs MF4CF
# =============================================================================
viewer_mf = show_dual_rgb(
    rgb_mf3,
    rgb_mf4,
    f'Model-Free — MF3CF (L) vs MF4CF (R)  (win={window_size})',
    left_label='MF3CF RGB',
    right_label='MF4CF RGB',
)
print('Viewer: MF3CF vs MF4CF')

---
## 5. Degree of Polarization (Barakat)

The Barakat scalar degree of polarisation measures how close a partially polarised wave
is to a fully polarised state, using the eigenvalues of $[T_3]$:

$$m = \sqrt{\max\!\left(1 - \frac{27\,\det[T_3]}{\operatorname{tr}[T_3]^3},\; 0\right)} \in [0, 1]$$

$m = 0$ is completely depolarised (entropy $H = 1$, random scatter);
$m = 1$ is fully polarised (rank-1 $[T_3]$, single scatterer).

**Reference**: Barakat, R. (1977). *Optica Acta*, 24(5), 537–540.


In [ ]:
# =============================================================================
# DegreeOfPolarization
# =============================================================================
dop_decomp = DegreeOfPolarization(window_size=window_size)
dop_c = dop_decomp.decompose_from_t3(t3)
print(f'DoP keys  : {list(dop_c.keys())}')
print(f'DoP range : [{dop_c["dop"].min():.4f}, {dop_c["dop"].max():.4f}]')
print(f'DoP mean  : {dop_c["dop"].mean():.4f}')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dop_c['dop'], cmap='gray', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, label='DoP  m  [0 = depol, 1 = fully pol]')
ax.set_title(f'Barakat Degree of Polarization  (win={window_size}, filter={filter_method!r})')
ax.axis('off')
plt.tight_layout()
plt.show()


---
## 6. Shannon Entropy

The polarimetric Shannon entropy (Morio et al. 2009) partitions the total scene
randomness into an intensity term $H_I$ and a polarimetric term $H_P$:

$$H_I = 3\ln\!\left(\frac{e\pi\,\operatorname{tr}[T_3]}{3}\right), \qquad
H_P = \ln(1 - m_B), \qquad H = H_I + H_P$$

where $m_B = 1 - 27\det[T_3]/\operatorname{tr}[T_3]^3$ is the Barakat intermediate
quantity (without the square root).  $H_I$ grows with total intensity; $H_P$ grows
with polarimetric disorder.

**Reference**: Morio, J., Réfrégier, P., Goudail, F., Dubois-Fernandez, P. C., & Dupuis, X. (2009). *IEEE GRSL*, 6(4), 728–732.


In [ ]:
# =============================================================================
# ShannonEntropy
# =============================================================================
se_decomp = ShannonEntropy(window_size=window_size)
se_c = se_decomp.decompose_from_t3(t3)
print(f'Shannon keys     : {list(se_c.keys())}')
for k in ('H_total', 'H_intensity', 'H_polarimetric'):
    arr = se_c[k]
    fin = arr[np.isfinite(arr)]
    print(f'  {k:18s}: mean={fin.mean():.3f}  range=[{fin.min():.3f}, {fin.max():.3f}]')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
panels_se = [
    ('H_total',        'Shannon Entropy  H',        'magma'),
    ('H_intensity',    'Intensity term  $H_I$',     'viridis'),
    ('H_polarimetric', 'Polarimetric term  $H_P$',  'plasma'),
]
for ax, (key, title, cmap) in zip(axes[:3], panels_se):
    arr = se_c[key]
    vmin, vmax = np.nanpercentile(arr, [2, 98])
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.axis('off')
axes[3].set_visible(False)
fig.suptitle(f'Shannon Entropy  (win={window_size}, filter={filter_method!r})', y=1.01)
plt.tight_layout()
plt.show()


---
## 7. Neumann De-orientation Decomposition

Neumann et al. (2010) apply a de-orientation rotation $\Phi$ to $[T_3]$ that removes
the effect of target orientation, then extract three coherence parameters from the
rotated matrix $[T_3']$:

| Parameter | Symbol | Range | Interpretation |
|---|---|---|---|
| Orientation angle | $\psi$ | $[-45°, 45°]$ | Surface tilt / azimuth orientation |
| Coherence magnitude | $\delta_\text{mod}$ | $[0, \infty)$ | Degree of coherence between HH+VV and HH−VV |
| Coherence phase | $\delta_\text{pha}$ | $[-180°, 180°]$ | Phase centre separation |
| Depolarisation | $\tau$ | $[0, 1]$ | 0 = no depolarisation, 1 = full |

$$\Phi = \tfrac{1}{4}\!\left(\pi + \arctan\!\frac{-2\,\operatorname{Re}(T_{23})}{T_{33} - T_{22}}\right)$$

**Reference**: Neumann, M., Ferro-Famil, L., & Reigber, A. (2010). *IEEE TGRS*, 48(3), 1227–1240.


In [ ]:
# =============================================================================
# NeumannDecomposition
# =============================================================================
neu_decomp = NeumannDecomposition(window_size=window_size)
neu_c = neu_decomp.decompose_from_t3(t3)
print(f'Neumann keys: {list(neu_c.keys())}')
for k in ('psi', 'delta_mod', 'delta_pha', 'tau'):
    arr = neu_c[k]
    print(f'  {k:12s}: range=[{arr.min():.3f}, {arr.max():.3f}]  mean={arr.mean():.3f}')

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
panels_neu = [
    ('psi',       'Orientation $\\psi$ (°)',               'RdBu_r',  (-45,  45)),
    ('delta_mod', 'Coherence $\\delta_\\mathrm{mod}$',     'hot',     None),
    ('delta_pha', 'Coh. phase $\\delta_\\mathrm{pha}$ (°)','hsv',     (-180, 180)),
    ('tau',       'Depolarisation $\\tau$',                 'viridis', (0,    1)),
]
for ax, (key, title, cmap, clim) in zip(axes, panels_neu):
    arr = neu_c[key]
    vmin = clim[0] if clim else np.nanpercentile(arr, 2)
    vmax = clim[1] if clim else np.nanpercentile(arr, 98)
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
fig.suptitle(f'Neumann Decomposition  (win={window_size}, filter={filter_method!r})', y=1.01)
plt.tight_layout()
plt.show()


---
## 8. Praks Parameters

Praks et al. (2009) define seven target-characterisation indices from the
span-normalised covariance matrix $M = [C_3]/\operatorname{span}$:

| Parameter | Formula | Range |
|---|---|---|
| Frobenius norm | $\|M\|_F^2 = \sum_{ij}|M_{ij}|^2$ | $[1/3,\, 1]$ |
| Scattering predominance | $\sqrt{\|M\|_F^2}$ | $[1/\sqrt{3},\, 1]$ |
| Scattering diversity | $1.5\,(1 - \|M\|_F^2)$ | $[0,\, 1]$ |
| Degree of purity | $2\sqrt{\max(\|M\|_F^2 - 1/4,\, 0)}$ | $[0,\, 1]$ |
| Depolarisation index | $1 - P_\text{pur}/\sqrt{3}$ | $[0,\, 1]$ |
| Alpha angle | $\arccos(\operatorname{Re}(M_{11}))$ | $[0°,\, 90°]$ |
| Entropy | $2.52 + 0.78\,\log|\!\det(M + 0.16 I)|/\log 3$ | — |

**Reference**: Praks, J., Hallikainen, M., & Antropov, O. (2009). *IEEE TGRS*, 47(11), 3811–3820.


In [ ]:
# =============================================================================
# PraksParameters
# =============================================================================
praks_decomp = PraksParameters(window_size=window_size)
praks_c = praks_decomp.decompose_from_c3(c3)
print(f'Praks keys: {list(praks_c.keys())}')
for k, v in praks_c.items():
    print(f'  {k:26s}: range=[{v.min():.4f}, {v.max():.4f}]  mean={v.mean():.4f}')

rgb_praks, _ = praks_decomp.to_rgb(praks_c)  # R=alpha, G=scatt_div, B=depol_idx

fig, axes = plt.subplots(4, 2, figsize=(12, 14))
axes = axes.ravel()
panels_praks = [
    ('alpha',                   'Alpha (°)',             'RdYlBu_r'),
    ('frobenius_norm',          'Frobenius norm',        'hot'),
    ('scattering_predominance', 'Scatt. predominance',   'viridis'),
    ('scattering_diversity',    'Scatt. diversity',      'plasma'),
    ('degree_purity',           'Degree of purity',      'magma'),
    ('depolarization_index',    'Depolarisation index',  'cividis'),
    ('entropy',                 'Entropy',               'inferno'),
]
for ax, (key, title, cmap) in zip(axes[:7], panels_praks):
    arr = praks_c[key]
    vmin, vmax = np.nanpercentile(arr, [2, 98])
    im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
axes[7].imshow(rgb_praks.transpose(1, 2, 0))
axes[7].set_title('RGB  (R=α, G=div, B=depol)', fontsize=9)
axes[7].axis('off')
fig.suptitle(f'Praks Parameters  (win={window_size}, filter={filter_method!r})')
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# GRDK viewer — Praks Parameters
# =============================================================================
viewer_praks = show_dual_rgb(
    rgb_raw,
    rgb_praks,
    f'Praks Parameters (win={window_size}) — Raw (L) vs Praks RGB (R)',
    left_label='Raw intensity RGB',
    right_label='Praks RGB',
)
print('Viewer: Praks Parameters')


---
## 9. Touzi TSVM Decomposition

The Target Scattering Vector Model (TSVM) of Touzi (2007) performs an
eigendecomposition of $[T_3]$ and extracts, for each eigenvector $\vec{k}_i$,
four physically interpretable parameters:

| Parameter | Symbol | Range | Meaning |
|---|---|---|---|
| Scattering type | $\alpha_i$ | $[0°, 90°]$ | $0°$ = odd-bounce, $90°$ = even-bounce |
| Scattering phase | $\phi_i$ | $(-180°, 180°]$ | Phase of dominant mechanism |
| Helicity | $\tau_i$ | $(-45°, 45°]$ | $0°$ = linear, $\pm 45°$ = circular |
| Orientation | $\psi_i$ | $(-90°, 90°]$ | Scatterer rotation in plane of incidence |

Probability-weighted means $\bar\alpha$, $\bar\phi$, $\bar\tau$, $\bar\psi$ summarise
across all three eigenvectors.

**Reference**: Touzi, R. (2007). *IEEE TGRS*, 45(9), 2941–2954.


In [ ]:
# =============================================================================
# TouziDecomposition
# =============================================================================
touzi_decomp = TouziDecomposition(window_size=window_size)
touzi_c = touzi_decomp.decompose_from_t3(t3)
print(f'Touzi keys ({len(touzi_c)}): {list(touzi_c.keys())}')
for k in ('alpha_mean', 'phi_mean', 'tau_mean', 'psi_mean'):
    fin = touzi_c[k][np.isfinite(touzi_c[k])]
    print(f'  {k:12s}: range=[{fin.min():.2f}°, {fin.max():.2f}°]  mean={fin.mean():.2f}°')

rgb_touzi, _ = touzi_decomp.to_rgb(touzi_c)  # R=ᾱ, G=|τ̄|, B=ψ̄

fig, axes = plt.subplots(4, 2, figsize=(12, 14))
axes = axes.ravel()
mean_panels = [
    ('alpha_mean', 'Mean $\\bar{\\alpha}$ (°)', 'RdYlBu_r', (0, 90)),
    ('phi_mean',   'Mean $\\bar{\\phi}$ (°)',   'hsv',      (-180, 180)),
    ('tau_mean',   'Mean $\\bar{\\tau}$ (°)',   'bwr',      (-45, 45)),
    ('psi_mean',   'Mean $\\bar{\\psi}$ (°)',   'RdBu_r',   (-90, 90)),
]
ev1_panels = [
    ('alpha1', 'EV-1  $\\alpha_1$ (°)', 'hot',    (0, 90)),
    ('tau1',   'EV-1  $\\tau_1$ (°)',   'bwr',    (-45, 45)),
    ('psi1',   'EV-1  $\\psi_1$ (°)',   'RdBu_r', (-90, 90)),
]
for ax, (key, title, cmap, clim) in zip(axes[:4], mean_panels):
    im = ax.imshow(touzi_c[key], cmap=cmap, vmin=clim[0], vmax=clim[1])
    fig.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
for ax, (key, title, cmap, clim) in zip(axes[4:7], ev1_panels):
    im = ax.imshow(touzi_c[key], cmap=cmap, vmin=clim[0], vmax=clim[1])
    fig.colorbar(im, ax=ax)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
axes[7].imshow(rgb_touzi.transpose(1, 2, 0))
axes[7].set_title('RGB  (R=$\\bar{\\alpha}$, G=$|\\bar{\\tau}|$, B=$\\bar{\\psi}$)', fontsize=9)
axes[7].axis('off')
fig.suptitle(f'Touzi TSVM  (win={window_size}, filter={filter_method!r})')
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# GRDK viewer — Touzi TSVM
# =============================================================================
viewer_touzi = show_dual_rgb(
    rgb_raw,
    rgb_touzi,
    f'Touzi TSVM (win={window_size}) — Raw (L) vs Touzi RGB (R)',
    left_label='Raw intensity RGB',
    right_label='Touzi RGB',
)
print('Viewer: Touzi TSVM')


---
## 10. Yamaguchi 4-Component Decomposition

Yamaguchi et al. separate total backscattered power into four physical mechanisms:

$$P_\text{span} = P_s + P_d + P_v + P_c$$

| Component | Symbol | Scatterer |
|---|---|---|
| Surface | $P_s$ | Single-bounce Bragg surface |
| Double-bounce | $P_d$ | Dihedral / corner reflector |
| Volume | $P_v$ | Randomly-oriented dipoles (vegetation) |
| Helix | $P_c$ | Circularly polarised component (urban asymmetry) |

Three model variants are available via `yamaguchi_model` in the Configuration cell:

- **`y4o`** — original Yamaguchi (2005): no rotation applied.
- **`y4r`** — rotation-corrected (2011): unitary rotation minimises $T_{23}$ before decomposition.
- **`y4s`** — extended volume model: Y4R with a volume-type selector.

The per-pixel decision loop is JIT-compiled with **numba** when `grdl[polsar]` is
installed (~300–700× faster than the equivalent pure-Python loop).

**References**:
Yamaguchi et al. (2005). *IEEE TGRS* 43(8), 1699–1706.
Yamaguchi et al. (2011). *IEEE TGRS* 49(6), 2251–2258.


In [ ]:
# =============================================================================
# Yamaguchi4C
# =============================================================================
import warnings

yam_decomp = Yamaguchi4C(window_size=window_size, model=yamaguchi_model)
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter('always')
    yam_c = yam_decomp.decompose_from_t3(t3)
for w in _w:
    print(f'[warning] {w.category.__name__}: {w.message}')

span = yam_c['span']
print(f'Yamaguchi4C model={yamaguchi_model!r}  keys: {list(yam_c.keys())}')
for k in ('surface', 'double_bounce', 'volume', 'helix'):
    arr = yam_c[k]
    frac = np.nanmean(arr) / (np.nanmean(span) + 1e-10)
    print(f'  {k:14s}: mean={np.nanmean(arr):.4f}  ({100*frac:.1f}% of mean span)')

rgb_yam, _ = yam_decomp.to_rgb(yam_c)  # R=Pd, G=Pv, B=Ps

fig, axes = plt.subplots(3, 2, figsize=(12, 12))
axes = axes.ravel()
components_yam = [
    ('surface',       'Surface $P_s$',       'Blues_r'),
    ('double_bounce', 'Double-bounce $P_d$',  'Reds_r'),
    ('volume',        'Volume $P_v$',         'Greens_r'),
    ('helix',         'Helix $P_c$',          'Purples_r'),
]
for ax, (key, title, cmap) in zip(axes[:4], components_yam):
    arr = yam_c[key]
    with np.errstate(divide='ignore', invalid='ignore'):
        arr_db = 10.0 * np.log10(np.maximum(arr, 1e-10))
    vmin, vmax = np.nanpercentile(arr_db, [5, 95])
    im = ax.imshow(arr_db, cmap=cmap, vmin=vmin, vmax=vmax)
    fig.colorbar(im, ax=ax, label='dB')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
axes[4].imshow(rgb_yam.transpose(1, 2, 0))
axes[4].set_title('RGB  (R=$P_d$, G=$P_v$, B=$P_s$)', fontsize=9)
axes[4].axis('off')
axes[5].set_visible(False)
fig.suptitle(
    f'Yamaguchi 4C [{yamaguchi_model.upper()}]  (win={window_size}, filter={filter_method!r})'
)
plt.tight_layout()
plt.show()


In [ ]:
# =============================================================================
# GRDK viewer — Yamaguchi 4C
# =============================================================================
viewer_yam = show_dual_rgb(
    rgb_raw,
    rgb_yam,
    f'Yamaguchi 4C [{yamaguchi_model.upper()}] (win={window_size}) — Raw (L) vs Yamaguchi RGB (R)',
    left_label='Raw intensity RGB',
    right_label='Yamaguchi RGB',
)
print(f'Viewer: Yamaguchi4C [{yamaguchi_model}]')


---
## 11. Vegetation Indices — RVI, GRVI, PRVI (Placeholder)

Polarimetric vegetation indices use the polarimetric entropy, span, and eigenvalues to
estimate vegetation cover, canopy structure, and biomass.

| Index | Formula | Reference |
|-------|---------|----------|
| RVI   | $4\sigma_{HV}^0 / (\sigma_{HH}^0 + \sigma_{VV}^0 + 2\sigma_{HV}^0)$ | Kim & van Zyl 2009 |
| GRVI  | $1 - GD_{VI}$ | Chandrasekar et al. 2010 |
| PRVI  | $(1 - \lambda_1/(\lambda_1+\lambda_2+\lambda_3)) \cdot \sigma_{VV}^0$ | Raney et al. 2011 |

**Not yet implemented** (Phase 6 of the PolSAR expansion plan).


In [ ]:
# =============================================================================
# Vegetation Indices — placeholder
# =============================================================================

# from grdl.image_processing.decomposition import RVI, GRVI, PRVI
#
# rvi  = RVI(window_size=window_size).decompose(shh, shv, svh, svv)
# grvi = GRVI(window_size=window_size).decompose(shh, shv, svh, svv)
# prvi = PRVI(window_size=window_size).decompose(shh, shv, svh, svv)
#
# fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# for ax, (name, data) in zip(axes, [('RVI', rvi['rvi']), ('GRVI', grvi['grvi']), ('PRVI', prvi['prvi'])]):
#     ax.imshow(data, vmin=0, vmax=1, cmap='YlGn')
#     ax.set_title(name)
#     ax.axis('off')
# plt.tight_layout()
# plt.show()

print('Vegetation indices (RVI, GRVI, PRVI) not yet implemented — placeholder only')